In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

T = "../results/tables/"
files = {
    "TF-IDF + LR": "tfidf_lr_v2_test_probs.csv",
    "DistilBERT": "distilbert_v2_test_probs.csv",
    "Qwen LoRA": "lora_test_probs.csv",
    "Qwen QLoRA": "qlora_test_probs.csv",
}
frames = {name: pd.read_csv(T + f).set_index("id") for name, f in files.items()}

ids = frames["DistilBERT"].index
y = frames["DistilBERT"]["y"].values
source = frames["DistilBERT"].loc[ids, "source_dataset"].values
probs = {}
for name, f in frames.items():
    assert set(f.index) == set(ids), name + ": different rows from the others"
    assert (f.loc[ids, "y"].values == y).all(), name + ": labels differ"
    probs[name] = f.loc[ids, "prob_attack"].values

meta = pd.read_csv("../data/processed/dataset_v2.csv", dtype=str, keep_default_na=False).set_index("id")
assert (meta.loc[ids, "split"] == "test").all(), "a non-test row is in the files"
group = meta.loc[ids, "group"].values

thr = pd.read_csv(T + "thresholds_from_validation.csv").set_index("model")["threshold_1pct_fp"].to_dict()
print("test rows:", len(ids), "| attacks:", int(y.sum()), "| safe:", int((y == 0).sum()), "| prompt families:", len(set(group)))
print("frozen cut-offs:", thr)

test rows: 3300 | attacks: 671 | safe: 2629 | prompt families: 2510
frozen cut-offs: {'TF-IDF + LR': 0.3548, 'DistilBERT': 0.7821, 'Qwen LoRA': 0.5115, 'Qwen QLoRA': 0.7834}


In [2]:
def counts(y, flag, w):
    tp = (w * y * flag).sum()
    fp = (w * (1 - y) * flag).sum()
    return tp, fp, (w * y).sum(), (w * (1 - y)).sum()


def op_metrics(y, flag, w):
    tp, fp, pos, neg = counts(y, flag, w)
    recall = tp / pos
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"recall": recall, "precision": precision, "f1": f1, "fp_rate": fp / neg, "safe_flagged": fp}


def recall_at_fp(y, p, rate):
    cut = np.quantile(p[y == 0], 1 - rate)
    return float((p[y == 1] > cut).mean())


ones = np.ones(len(y))
rows = []
for name, p in probs.items():
    m_cut = op_metrics(y, (p > thr[name]).astype(float), ones)
    m_def = op_metrics(y, (p >= 0.5).astype(float), ones)
    rows.append({
        "model": name,
        "AP": average_precision_score(y, p),
        "recall@val_cutoff": m_cut["recall"],
        "precision@val_cutoff": m_cut["precision"],
        "F1@val_cutoff": m_cut["f1"],
        "safe_flagged@val_cutoff": int(m_cut["safe_flagged"]),
        "FP_rate@val_cutoff": m_cut["fp_rate"],
        "recall@0.5": m_def["recall"],
        "precision@0.5": m_def["precision"],
        "F1@0.5": m_def["f1"],
        "safe_flagged@0.5": int(m_def["safe_flagged"]),
        "recall@1%FP(test)": recall_at_fp(y, p, 0.01),
    })
point_raw = pd.DataFrame(rows)
point = point_raw.round(3)
point.to_csv(T + "test_results_point.csv", index=False)
print(point.set_index("model").T.to_string())

rows = []
for name in probs:
    flag = probs[name] > thr[name]
    for s in sorted(set(source)):
        m = source == s
        att = m & (y == 1)
        saf = m & (y == 0)
        rows.append({
            "model": name, "source": s,
            "attack_rows": int(att.sum()),
            "attacks_caught": round(float(flag[att].mean()), 3) if att.sum() else np.nan,
            "safe_rows": int(saf.sum()),
            "safe_flagged_rate": round(float(flag[saf].mean()), 3) if saf.sum() else np.nan,
        })
by_source = pd.DataFrame(rows)
by_source.to_csv(T + "test_by_source.csv", index=False)
print()
print(by_source.to_string(index=False))

model                    TF-IDF + LR  DistilBERT  Qwen LoRA  Qwen QLoRA
AP                             0.949       0.965      0.971       0.972
recall@val_cutoff              0.851       0.855      0.894       0.885
precision@val_cutoff           0.928       0.936      0.946       0.952
F1@val_cutoff                  0.888       0.894      0.920       0.917
safe_flagged@val_cutoff       44.000      39.000     34.000      30.000
FP_rate@val_cutoff             0.017       0.015      0.013       0.011
recall@0.5                     0.808       0.912      0.894       0.917
precision@0.5                  0.966       0.895      0.945       0.928
F1@0.5                         0.880       0.903      0.919       0.922
safe_flagged@0.5              19.000      72.000     35.000      48.000
recall@1%FP(test)              0.829       0.805      0.878       0.866

      model        source  attack_rows  attacks_caught  safe_rows  safe_flagged_rate
TF-IDF + LR       deepset           28           0

In [3]:
rng = np.random.default_rng(42)
codes, uniques = pd.factorize(group)
n_groups = len(uniques)
B = 1000
names = list(probs)
store = {n: {"AP": [], "recall": [], "precision": [], "fp_rate": []} for n in names}

for _ in range(B):
    draw = rng.integers(0, n_groups, n_groups)
    w = np.bincount(draw, minlength=n_groups)[codes].astype(float)
    for n in names:
        p = probs[n]
        m = op_metrics(y, (p > thr[n]).astype(float), w)
        store[n]["AP"].append(average_precision_score(y, p, sample_weight=w))
        store[n]["recall"].append(m["recall"])
        store[n]["precision"].append(m["precision"])
        store[n]["fp_rate"].append(m["fp_rate"])
store = {n: {k: np.array(v) for k, v in d.items()} for n, d in store.items()}


def ci(a):
    return np.percentile(a, [2.5, 97.5])


labels = {"AP": "AP", "recall": "recall@val_cutoff",
          "precision": "precision@val_cutoff", "fp_rate": "FP_rate@val_cutoff"}
raw = point_raw.set_index("model")
rows = []
for n in names:
    r = {"model": n}
    for k, label in labels.items():
        lo, hi = ci(store[n][k])
        r[label] = f"{raw.loc[n, label]:.3f} [{lo:.3f}, {hi:.3f}]"
    rows.append(r)
ci_table = pd.DataFrame(rows)
ci_table.to_csv(T + "test_results_ci.csv", index=False)
print("95% confidence intervals (point estimate [low, high]), prompt families redrawn", B, "times")
print(ci_table.to_string(index=False))

pairs = [("Qwen LoRA", "DistilBERT"), ("Qwen QLoRA", "DistilBERT"), ("Qwen LoRA", "Qwen QLoRA"),
         ("Qwen LoRA", "TF-IDF + LR"), ("DistilBERT", "TF-IDF + LR")]
rows = []
for a, b in pairs:
    for k in ["AP", "recall"]:
        d = store[a][k] - store[b][k]
        lo, hi = ci(d)
        est = raw.loc[a, labels[k]] - raw.loc[b, labels[k]]
        rows.append({
            "comparison": f"{a} minus {b}", "metric": labels[k],
            "difference": round(float(est), 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "CI excludes 0": bool(lo > 0 or hi < 0),
        })
diffs = pd.DataFrame(rows)
diffs.to_csv(T + "test_paired_differences.csv", index=False)
print()
print(diffs.to_string(index=False))

95% confidence intervals (point estimate [low, high]), prompt families redrawn 1000 times
      model                   AP    recall@val_cutoff precision@val_cutoff   FP_rate@val_cutoff
TF-IDF + LR 0.949 [0.922, 0.968] 0.851 [0.798, 0.894] 0.928 [0.895, 0.954] 0.017 [0.011, 0.023]
 DistilBERT 0.965 [0.947, 0.978] 0.855 [0.808, 0.898] 0.936 [0.906, 0.958] 0.015 [0.010, 0.021]
  Qwen LoRA 0.971 [0.957, 0.983] 0.894 [0.854, 0.929] 0.946 [0.924, 0.965] 0.013 [0.008, 0.019]
 Qwen QLoRA 0.972 [0.958, 0.982] 0.885 [0.841, 0.920] 0.952 [0.931, 0.968] 0.011 [0.008, 0.016]

                  comparison            metric  difference          95% CI  CI excludes 0
  Qwen LoRA minus DistilBERT                AP       0.007 [-0.000, 0.014]          False
  Qwen LoRA minus DistilBERT recall@val_cutoff       0.039  [0.015, 0.067]           True
 Qwen QLoRA minus DistilBERT                AP       0.007 [-0.000, 0.017]          False
 Qwen QLoRA minus DistilBERT recall@val_cutoff       0.030  [0.007, 0

In [1]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

root = Path("..").resolve()
print("project root:", root)
print("python:", sys.version.split()[0])

print("\ntop level:")
for p in sorted(root.iterdir()):
    if not p.name.startswith("."):
        print("  ", p.name + ("/" if p.is_dir() else ""))

model_dir = root / "models" / "distilbert_v2"
print("\nDistilBERT folder exists:", model_dir.exists())
if model_dir.exists():
    for f in sorted(model_dir.iterdir()):
        print(f"   {f.name:35s} {f.stat().st_size / 1e6:8.1f} MB")

thr = root / "results" / "tables" / "thresholds_from_validation.csv"
print("\nthresholds file exists:", thr.exists())
if thr.exists():
    print(thr.read_text())

print("library versions:")
for lib in ["streamlit", "torch", "transformers", "numpy", "pandas", "huggingface_hub"]:
    try:
        print("  ", lib, version(lib))
    except PackageNotFoundError:
        print("  ", lib, "NOT installed")

project root: /Users/parth/projects/prompt-guard-detector
python: 3.11.18

top level:
   configs/
   data/
   docs/
   models/
   notebooks/
   results/
   scripts/
   src/

DistilBERT folder exists: True
   config.json                              0.0 MB
   model.safetensors                      267.8 MB
   tokenizer.json                           0.7 MB
   tokenizer_config.json                    0.0 MB

thresholds file exists: True
model,threshold_1pct_fp,val_recall,val_precision,val_safe_flagged
TF-IDF + LR,0.3548,0.815,0.953,27
DistilBERT,0.7821,0.846,0.954,27
Qwen LoRA,0.5115,0.859,0.955,27
Qwen QLoRA,0.7834,0.849,0.955,27

library versions:
   streamlit NOT installed
   torch 2.15.0
   transformers 5.23.0
   numpy 2.4.6
   pandas 3.1.1
   huggingface_hub 2.4.0
